# 🚀 AIOps Knowledge Check — Lê Kim Dung

**Ngày nộp:** 01/06/2026  
**Repository:** https://github.com/KimDung1/aiops_lekimdung

---

## Mục lục
1. [Câu 1: Skewness & 3σ](#câu-1)
2. [Câu 2: So sánh 3σ vs EWMA vs STL](#câu-2)
3. [Câu 3: Isolation Forest](#câu-3)
4. [Câu 4: Univariate vs Multivariate](#câu-4)
5. [Câu 5: Precision vs Recall trong AIOps](#câu-5)
6. [Ảnh Knowledge Check Viết Tay](#ảnh-viết-tay)


In [ ]:
# ── Cài đặt thư viện cần thiết ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Seed để tái lập kết quả
np.random.seed(42)

print('✅ Libraries loaded successfully!')
print(f'NumPy: {np.__version__}')
print(f'Pandas: {pd.__version__}')


---
## Câu 1: Skewness là gì? 3σ sai ở đâu? Và 2 cách xử lý
<a id='câu-1'></a>


### 1.1 Skewness là gì?

**Skewness** (độ lệch) là thống kê đo mức độ bất đối xứng của phân phối dữ liệu so với phân phối chuẩn (Normal Distribution).

- **Right-skewed (Positive Skew):** Đuôi dài về phía phải. `mean > median > mode`
  - Ví dụ: response time, request latency, income distribution
- **Left-skewed (Negative Skew):** Đuôi dài về phía trái. `mean < median < mode`
  - Ví dụ: test scores near maximum, time-to-failure after maintenance
- **Skewness = 0:** Phân phối đối xứng (Normal Distribution)

Công thức: $\text{Skewness} = \frac{E[(X-\mu)^3]}{\sigma^3}$


In [ ]:
# ── Demo: Visualize Right-Skewed Data ──
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Câu 1: Skewness - Phân phối dữ liệu', fontsize=14, fontweight='bold')

# Left skew
left_skew = np.concatenate([np.random.normal(90, 5, 900), np.random.uniform(50, 75, 100)])
axes[0].hist(left_skew, bins=40, color='#E74C3C', alpha=0.7, edgecolor='white')
axes[0].axvline(np.mean(left_skew), color='blue', linestyle='--', label=f'Mean={np.mean(left_skew):.1f}', linewidth=2)
axes[0].axvline(np.median(left_skew), color='green', linestyle='-', label=f'Median={np.median(left_skew):.1f}', linewidth=2)
axes[0].set_title(f'Left Skew (skewness={stats.skew(left_skew):.2f})')
axes[0].legend(); axes[0].set_xlabel('Value')

# Normal
normal = np.random.normal(70, 10, 1000)
axes[1].hist(normal, bins=40, color='#2ECC71', alpha=0.7, edgecolor='white')
axes[1].axvline(np.mean(normal), color='blue', linestyle='--', label=f'Mean={np.mean(normal):.1f}', linewidth=2)
axes[1].axvline(np.median(normal), color='green', linestyle='-', label=f'Median={np.median(normal):.1f}', linewidth=2)
axes[1].set_title(f'Normal (skewness={stats.skew(normal):.2f})')
axes[1].legend(); axes[1].set_xlabel('Value')

# Right skew (response time)
right_skew = np.concatenate([np.random.exponential(scale=50, size=950), np.random.uniform(400, 800, 50)])
axes[2].hist(right_skew, bins=40, color='#3498DB', alpha=0.7, edgecolor='white')
axes[2].axvline(np.mean(right_skew), color='blue', linestyle='--', label=f'Mean={np.mean(right_skew):.1f}', linewidth=2)
axes[2].axvline(np.median(right_skew), color='green', linestyle='-', label=f'Median={np.median(right_skew):.1f}', linewidth=2)
axes[2].set_title(f'Right Skew (skewness={stats.skew(right_skew):.2f})')
axes[2].legend(); axes[2].set_xlabel('Response Time (ms)')

plt.tight_layout()
plt.savefig('images/q1_skewness_types.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Right skew: skewness = {stats.skew(right_skew):.3f}')


### 1.2 3σ sai ở đâu khi data bị skew?

**3-Sigma (3σ) rule** giả định data tuân theo **Normal Distribution**:
- `upper_threshold = mean + 3σ`
- `lower_threshold = mean - 3σ`

**Vấn đề khi data bị skew (right-skewed):**

| Vấn đề | Mô tả |
|--------|-------|
| Mean bị kéo lên | Mean > Median → threshold cả hai phía đều lệch |
| False Positives cao | Phía trái (đuôi ngắn) → threshold thấp hơn thực tế → flag normal values |
| False Negatives cao | Phía phải (đuôi dài) → threshold quá cao → miss real anomalies |
| Asymmetric bounds | 3σ tạo bounds đối xứng nhưng data không đối xứng |


In [ ]:
# ── Demo: 3σ fail với right-skewed response time ──
np.random.seed(42)
response_times = np.concatenate([
    np.random.exponential(scale=50, size=970),   # normal traffic
    np.random.uniform(400, 600, 30)               # actual anomalies
])

mean = np.mean(response_times)
std = np.std(response_times)
upper_3sigma = mean + 3 * std
lower_3sigma = mean - 3 * std

# Robust method: Median + IQR
median = np.median(response_times)
iqr = stats.iqr(response_times)
upper_robust = median + 3 * (iqr / 1.35)
lower_robust = median - 3 * (iqr / 1.35)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('3σ vs Robust Method trên Right-Skewed Data', fontsize=13, fontweight='bold')

for ax, title, upper, lower, color, label in [
    (axes[0], '3σ Method (FAIL on skewed data)', upper_3sigma, lower_3sigma, '#E74C3C', '3σ threshold'),
    (axes[1], 'Robust: Median + IQR (CORRECT)', upper_robust, lower_robust, '#2ECC71', 'IQR threshold')
]:
    ax.hist(response_times, bins=50, color='#3498DB', alpha=0.6, edgecolor='white')
    ax.axvline(upper, color=color, linestyle='--', linewidth=2.5, label=f'{label}: {upper:.1f}ms')
    ax.axvline(lower, color=color, linestyle=':', linewidth=2.5, label=f'Lower: {lower:.1f}ms')
    ax.axvline(mean if 'σ' in title else median, color='orange', linestyle='-',
               linewidth=2, label=f"{'Mean' if '3σ' in title else 'Median'}: {mean:.1f if '3σ' in title else median:.1f}ms")
    ax.fill_betweenx([0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 300],
                      upper, max(response_times), color=color, alpha=0.15, label='Anomaly zone')
    ax.set_title(title); ax.legend(fontsize=8); ax.set_xlabel('Response Time (ms)')

# Fix y-limits after fill
for ax in axes:
    ax.set_ylim(0, 300)
    y_max = 300
    ax.collections[0].set_clim(0, y_max) if ax.collections else None

plt.tight_layout()
plt.savefig('images/q1_3sigma_fail.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'3σ Upper threshold: {upper_3sigma:.2f} ms')
print(f'Robust Upper threshold: {upper_robust:.2f} ms')
fp_3sigma = np.sum(response_times[:970] > upper_3sigma)
fp_robust = np.sum(response_times[:970] > upper_robust)
print(f'False Positives (3σ): {fp_3sigma}')
print(f'False Positives (Robust): {fp_robust}')


### 1.3 Hai cách xử lý data skewed

**Cách 1: Log Transformation**
$$x' = \log(x + 1)$$
- Compress đuôi dài của right-skewed distribution về gần Normal
- Dùng `log(x+1)` để tránh log(0) = -∞
- Sau transform: có thể áp dụng 3σ bình thường

**Cách 2: Robust Statistics (Median + IQR)**
$$\text{threshold} = \text{Median} \pm 3 \times \frac{\text{IQR}}{1.35}$$
- IQR/1.35 ≈ σ cho Normal Distribution
- Median và IQR không bị ảnh hưởng bởi outliers
- Không cần transform data


In [ ]:
# ── Demo: 2 cách xử lý skewed data ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Câu 1: Hai Cách Xử Lý Skewed Data', fontsize=14, fontweight='bold')

# Original data
rt = response_times.copy()

# === Cách 1: Log Transformation ===
rt_log = np.log1p(rt)
mean_log = np.mean(rt_log); std_log = np.std(rt_log)
upper_log = mean_log + 3 * std_log; lower_log = mean_log - 3 * std_log

axes[0, 0].hist(rt, bins=50, color='#E74C3C', alpha=0.7)
axes[0, 0].set_title('Original Data (Right-Skewed)', fontweight='bold')
axes[0, 0].set_xlabel('Response Time (ms)')
axes[0, 0].annotate(f'Skewness: {stats.skew(rt):.2f}', xy=(0.7, 0.85), xycoords='axes fraction', fontsize=11)

axes[0, 1].hist(rt_log, bins=50, color='#2ECC71', alpha=0.7)
axes[0, 1].axvline(upper_log, color='red', linestyle='--', linewidth=2, label=f'3σ upper: {upper_log:.2f}')
axes[0, 1].axvline(lower_log, color='red', linestyle=':', linewidth=2, label=f'3σ lower: {lower_log:.2f}')
axes[0, 1].set_title('Sau Log Transformation → Near Normal', fontweight='bold')
axes[0, 1].set_xlabel('log(Response Time + 1)')
axes[0, 1].legend()
axes[0, 1].annotate(f'Skewness: {stats.skew(rt_log):.2f}', xy=(0.05, 0.85), xycoords='axes fraction', fontsize=11)

# === Cách 2: Robust IQR ===
q1, q3 = np.percentile(rt, [25, 75])
iqr_val = q3 - q1
upper_iqr = np.median(rt) + 3 * (iqr_val / 1.35)
lower_iqr = max(0, np.median(rt) - 3 * (iqr_val / 1.35))

axes[1, 0].hist(rt, bins=50, color='#9B59B6', alpha=0.7)
axes[1, 0].axvline(upper_iqr, color='red', linestyle='--', linewidth=2.5, label=f'Robust upper: {upper_iqr:.1f}ms')
axes[1, 0].axvline(np.median(rt), color='orange', linestyle='-', linewidth=2, label=f'Median: {np.median(rt):.1f}ms')
axes[1, 0].set_title('Cách 2: Robust Median + IQR', fontweight='bold')
axes[1, 0].set_xlabel('Response Time (ms)'); axes[1, 0].legend()

# Comparison table
axes[1, 1].axis('off')
table_data = [
    ['Phương pháp', 'Upper Threshold', 'FP Count', 'FN Count'],
    ['3σ (Original)', f'{upper_3sigma:.1f}ms', str(fp_3sigma), '~0 (too high)'],
    ['Log + 3σ', f'{np.expm1(upper_log):.1f}ms', str(np.sum(rt[:970] > np.expm1(upper_log))), str(np.sum(rt[970:] < np.expm1(upper_log)))],
    ['Median + IQR', f'{upper_iqr:.1f}ms', str(np.sum(rt[:970] > upper_iqr)), str(np.sum(rt[970:] < upper_iqr))]
]
tbl = axes[1, 1].table(cellText=table_data[1:], colLabels=table_data[0],
                        loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.2, 2)
axes[1, 1].set_title('So sánh kết quả', fontweight='bold')

plt.tight_layout()
plt.savefig('images/q1_solutions.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Câu 1 hoàn thành!')


---
## Câu 2: So sánh 3σ vs EWMA vs STL
<a id='câu-2'></a>

| | **3σ** | **EWMA** | **STL** |
|---|---|---|---|
| **Detect loại anomaly** | Point anomaly, sudden spike trên stable data | Gradual drift, slow trend shift | Seasonal anomaly, deviation từ periodic pattern |
| **Fail ở đâu** | Data skewed, seasonal data (mùa vụ), concept drift | Sudden spike (threshold thích nghi chậm), cold start, mất memory ngắn | Non-periodic data, period thay đổi, too many parameters |
| **Dùng khi nào** | Stable metrics, normal distribution, simple threshold | Slow-changing metrics, need weighted recent | Clear seasonality (daily/weekly pattern) |

### Chi tiết từng phương pháp:

**3σ (Three-Sigma Rule):**
- Nguyên lý: 99.7% dữ liệu Normal nằm trong ±3σ
- Static threshold: không thích nghi với trend
- Rất nhanh, đơn giản, không cần training

**EWMA (Exponentially Weighted Moving Average):**
- Nguyên lý: $\text{EWMA}_t = \lambda \cdot x_t + (1-\lambda) \cdot \text{EWMA}_{t-1}$
- λ (smoothing factor): cao → phản ứng nhanh hơn với thay đổi
- Fail với sudden spike vì moving average bị kéo dần về phía spike → threshold tăng theo

**STL (Seasonal-Trend decomposition using LOESS):**
- Tách: `Original = Trend + Seasonal + Residual`
- Detect anomaly trên **Residual** sau khi loại bỏ trend và seasonality
- Cần ≥2 chu kỳ đầy đủ để xác định pattern


In [ ]:
# ── Demo: So sánh 3 phương pháp trên time-series có seasonality ──
np.random.seed(42)
n = 200
t = np.arange(n)

# Tạo time series: trend + seasonal + noise + anomalies
trend = 0.05 * t
seasonal = 20 * np.sin(2 * np.pi * t / 24)  # chu kỳ 24 điểm
noise = np.random.normal(0, 3, n)
signal = trend + seasonal + noise + 100

# Inject anomalies
anomaly_idx = [50, 100, 150, 175]
anomaly_types = ['Sudden Spike', 'Gradual (EWMA miss)', 'Seasonal False FP', 'True Anomaly']
signal[50] += 40        # sudden spike
signal[90:105] += np.linspace(0, 15, 15)  # gradual drift
signal[150] += 30       # seasonal peak (might be false positive for 3σ)
signal[175] += 35       # true anomaly mid-cycle

# === 3σ Method ===
mean_3s = np.mean(signal)
std_3s = np.std(signal)
upper_3s = mean_3s + 3 * std_3s
lower_3s = mean_3s - 3 * std_3s
anomalies_3s = np.where((signal > upper_3s) | (signal < lower_3s))[0]

# === EWMA Method ===
lam = 0.1
ewma = np.zeros(n)
ewma[0] = signal[0]
for i in range(1, n):
    ewma[i] = lam * signal[i] + (1 - lam) * ewma[i-1]
ewma_std = np.std(signal[:50])  # estimate from initial period
upper_ewma = ewma + 2.5 * ewma_std
lower_ewma = ewma - 2.5 * ewma_std
anomalies_ewma = np.where((signal > upper_ewma) | (signal < lower_ewma))[0]

# === STL-like: Simple Seasonal Decomposition ===
period = 24
seasonal_component = np.array([np.mean(signal[i::period]) for i in range(period)])
seasonal_full = np.tile(seasonal_component, n // period + 1)[:n]
deseasonalized = signal - seasonal_full
trend_component = pd.Series(deseasonalized).rolling(window=12, center=True).mean().fillna(method='bfill').fillna(method='ffill').values
residual = deseasonalized - trend_component
res_std = np.std(residual)
anomalies_stl = np.where(np.abs(residual) > 2.5 * res_std)[0]

# Plot
fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=True)
fig.suptitle('Câu 2: So sánh 3σ vs EWMA vs STL', fontsize=14, fontweight='bold')

colors = ['#E74C3C', '#F39C12', '#2ECC71']
methods = [
    ('3σ Method', upper_3s * np.ones(n), lower_3s * np.ones(n), anomalies_3s, colors[0]),
    ('EWMA Method', upper_ewma, lower_ewma, anomalies_ewma, colors[1]),
    ('STL Method (Residual)', 2.5 * res_std * np.ones(n), -2.5 * res_std * np.ones(n), anomalies_stl, colors[2])
]

for ax, (name, upper, lower, anom, c) in zip(axes, methods):
    data_to_plot = signal if name != 'STL Method (Residual)' else residual
    ax.plot(t, data_to_plot, color='#3498DB', linewidth=1, alpha=0.8, label='Signal')
    ax.fill_between(t, upper, lower, alpha=0.15, color=c, label='Normal band')
    ax.plot(t, upper, color=c, linestyle='--', linewidth=1.5, alpha=0.7)
    ax.plot(t, lower, color=c, linestyle='--', linewidth=1.5, alpha=0.7)
    if len(anom) > 0:
        ax.scatter(anom, data_to_plot[anom], color='red', s=80, zorder=5, label=f'Detected ({len(anom)})')
    ax.set_title(f'{name} — Detected: {len(anom)} anomalies', fontweight='bold')
    ax.legend(loc='upper left', fontsize=9); ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time Step')
plt.tight_layout()
plt.savefig('images/q2_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'3σ detected: {len(anomalies_3s)} | EWMA detected: {len(anomalies_ewma)} | STL detected: {len(anomalies_stl)}')
print('✅ Câu 2 hoàn thành!')


---
## Câu 3: Isolation Forest — Path Length Ngắn = Anomaly
<a id='câu-3'></a>

### 3.1 Ý tưởng cốt lõi

**Isolation Forest** dựa trên nguyên lý: **Anomalies dễ bị isolate hơn normal points.**

**Quy trình:**
1. Xây dựng nhiều **Isolation Trees** (random binary trees)
2. Mỗi node: **random chọn feature** → **random chọn split value** trong range của feature đó
3. Tiếp tục split cho đến khi mỗi điểm bị isolate (1 điểm mỗi leaf)

**Tại sao path ngắn = anomaly?**
- **Anomaly:** Nằm xa cluster, ít điểm lân cận → chỉ cần **ít lần split** để isolate → **path length ngắn**
- **Normal point:** Nằm trong cluster dày đặc → cần **nhiều lần split** để tách ra → **path length dài**

**Anomaly Score:**
$$s(x, n) = 2^{-\frac{E[h(x)]}{c(n)}}$$
- $E[h(x)]$: expected path length của điểm x
- $c(n)$: average path length của unsuccessful BST search (normalization)
- $s \to 1$: anomaly | $s \approx 0.5$: normal | $s \to 0$: definitely normal


In [ ]:
# ── Demo: Isolation Forest Path Length Visualization ──
from sklearn.ensemble import IsolationForest

np.random.seed(42)

# Tạo dataset 2D
X_normal = np.random.multivariate_normal([0, 0], [[1, 0.5], [0.5, 1]], 300)
X_anomaly = np.array([[5, 5], [-5, 4], [4, -4], [6, -2], [-4, -5]])
X = np.vstack([X_normal, X_anomaly])
labels = np.array([0]*300 + [1]*5)  # 0=normal, 1=anomaly

# Train Isolation Forest
iforest = IsolationForest(n_estimators=100, contamination=0.02, random_state=42)
iforest.fit(X)
scores = iforest.score_samples(X)
predictions = iforest.predict(X)  # -1=anomaly, 1=normal

# Anomaly score (convert to 0-1 range)
anomaly_scores = -scores  # higher = more anomalous

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Câu 3: Isolation Forest — Path Length Visualization', fontsize=13, fontweight='bold')

# Plot 1: Data distribution
axes[0].scatter(X_normal[:, 0], X_normal[:, 1], c='#3498DB', s=20, alpha=0.5, label='Normal')
axes[0].scatter(X_anomaly[:, 0], X_anomaly[:, 1], c='red', s=100, marker='*', zorder=5, label='Anomaly')
axes[0].set_title('Data Distribution\n(Normal cluster + Anomalies)'); axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Anomaly scores heatmap
xx, yy = np.meshgrid(np.linspace(-8, 8, 100), np.linspace(-8, 8, 100))
Z = iforest.score_samples(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
axes[1].contourf(xx, yy, Z, levels=20, cmap='RdYlGn')
axes[1].scatter(X_normal[:, 0], X_normal[:, 1], c='white', s=15, alpha=0.4)
axes[1].scatter(X_anomaly[:, 0], X_anomaly[:, 1], c='black', s=150, marker='*', zorder=5, label='True Anomaly')
axes[1].set_title('Anomaly Score Heatmap\n(Red=Anomaly, Green=Normal)'); axes[1].legend()

# Plot 3: Score distribution
normal_scores = anomaly_scores[labels == 0]
anom_scores = anomaly_scores[labels == 1]
axes[2].hist(normal_scores, bins=30, color='#2ECC71', alpha=0.7, label='Normal points')
axes[2].hist(anom_scores, bins=5, color='#E74C3C', alpha=0.9, label='Anomaly points')
axes[2].axvline(np.percentile(anomaly_scores, 98), color='black', linestyle='--',
               linewidth=2, label='98th percentile threshold')
axes[2].set_title('Anomaly Score Distribution\n(Anomalies have higher scores)')
axes[2].set_xlabel('Anomaly Score (higher = more anomalous)')
axes[2].legend()

plt.tight_layout()
plt.savefig('images/q3_isolation_forest.png', dpi=120, bbox_inches='tight')
plt.show()

detected = np.sum(predictions[labels==1] == -1)
print(f'Anomalies detected: {detected}/5')
print(f'Anomaly scores: {anom_scores}')
print(f'Normal score range: [{normal_scores.min():.3f}, {normal_scores.max():.3f}]')


### 3.2 Tại sao cần Feature Engineering trước khi feed vào Isolation Forest?

| Vấn đề | Hậu quả nếu không xử lý | Giải pháp |
|--------|--------------------------|--------|
| **Curse of Dimensionality** | Features vô nghĩa → random splits không phân tách anomaly tốt | Feature selection, PCA |
| **Scale khác nhau** | Feature có range lớn dominance random split selection | MinMaxScaler / StandardScaler |
| **Time-series raw** | Timestamp vô nghĩa với iForest | Extract lag, rolling stats, hour-of-day |
| **Correlated features** | Nhiễu → path length không phản ánh isolation thực | Remove correlated features |


In [ ]:
# ── Demo: Feature Engineering impact on Isolation Forest ──
np.random.seed(42)
n_pts = 500

# Tạo time series metric (CPU usage)
t_series = np.arange(n_pts)
cpu = 50 + 20*np.sin(2*np.pi*t_series/24) + np.random.normal(0, 3, n_pts)
cpu[200] += 40   # anomaly spike
cpu[201] += 35
cpu[350:360] += np.linspace(0, 30, 10)  # gradual anomaly

# Method 1: Feed raw data only (bad)
X_raw = cpu.reshape(-1, 1)
iforest_raw = IsolationForest(contamination=0.03, random_state=42)
pred_raw = iforest_raw.fit_predict(X_raw)

# Method 2: Feature engineering (good)
df = pd.DataFrame({'cpu': cpu})
df['hour'] = t_series % 24
df['lag1'] = df['cpu'].shift(1).fillna(method='bfill')
df['lag2'] = df['cpu'].shift(2).fillna(method='bfill')
df['rolling_mean'] = df['cpu'].rolling(6, min_periods=1).mean()
df['rolling_std'] = df['cpu'].rolling(6, min_periods=1).std().fillna(0)
df['diff'] = df['cpu'].diff().fillna(0)
df['z_score'] = (df['cpu'] - df['rolling_mean']) / (df['rolling_std'] + 1e-6)

scaler = StandardScaler()
X_engineered = scaler.fit_transform(df.values)
iforest_eng = IsolationForest(contamination=0.03, random_state=42)
pred_eng = iforest_eng.fit_predict(X_engineered)

fig, axes = plt.subplots(2, 1, figsize=(15, 8))
fig.suptitle('Câu 3: Feature Engineering Impact on Isolation Forest', fontsize=13, fontweight='bold')

true_anomaly = np.zeros(n_pts); true_anomaly[200:202] = 1; true_anomaly[350:360] = 1

for ax, preds, title in [
    (axes[0], pred_raw, 'Không có Feature Engineering (Raw CPU only)'),
    (axes[1], pred_eng, 'Có Feature Engineering (Lag + Rolling + Z-score + Hour)')
]:
    ax.plot(t_series, cpu, color='#3498DB', linewidth=1, alpha=0.8, label='CPU %')
    detected_idx = np.where(preds == -1)[0]
    true_idx = np.where(true_anomaly == 1)[0]
    ax.scatter(detected_idx, cpu[detected_idx], color='red', s=40, zorder=5, label=f'Detected: {len(detected_idx)}')
    ax.scatter(true_idx, cpu[true_idx], color='orange', s=80, marker='^', zorder=4, label=f'True anomaly: {len(true_idx)}')
    
    tp = len(set(detected_idx) & set(true_idx))
    fp = len(detected_idx) - tp
    fn = len(true_idx) - tp
    ax.set_title(f'{title}\nTP={tp}, FP={fp}, FN={fn}', fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time Step')
plt.tight_layout()
plt.savefig('images/q3_feature_engineering.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Câu 3 hoàn thành!')


---
## Câu 4: Univariate vs Multivariate — Scenario: Memory Leak
<a id='câu-4'></a>

### Scenario: Memory Leak trong Production Service

Một service đang bị **memory leak** – không phải spike đột ngột mà là rò rỉ dần dần:
- Memory usage: tăng dần từ 60% → 85% theo thời gian
- CPU usage: tăng nhẹ vì GC chạy thường xuyên hơn
- GC pause time: tăng dần vì heap ngày càng lớn
- Response latency: tăng nhẹ vì GC pauses

**Quan trọng:** Mỗi metric riêng lẻ vẫn **dưới ngưỡng threshold!**

### Tại sao Univariate MISS?
- Phân tích từng metric riêng lẻ với threshold cố định (e.g., memory > 90%)
- Memory 85% < 90% threshold → **No alert**
- CPU 65% < 80% threshold → **No alert**  
- Latency 180ms < 300ms threshold → **No alert**
- **Không nhìn thấy PATTERN:** Tất cả cùng tăng đồng thời

### Tại sao Multivariate CATCH?
- Phân tích **correlation** và **joint distribution** của các metrics
- Detect pattern: Memory↑ & CPU↑ & GC↑ & Latency↑ cùng lúc
- Trong không gian nhiều chiều: điểm này **nằm xa cluster** của trạng thái bình thường
- Score anomaly cao → **Alert!**


In [ ]:
# ── Demo: Memory Leak - Univariate miss, Multivariate catch ──
np.random.seed(42)
n_time = 100
t_arr = np.arange(n_time)

# Simulate metrics (first 60 = normal, 60-100 = memory leak phase)
memory = np.concatenate([
    60 + np.random.normal(0, 2, 60),  # normal: ~60%
    60 + np.linspace(0, 25, 40) + np.random.normal(0, 1.5, 40)  # leak: 60→85%
])
cpu = np.concatenate([
    45 + np.random.normal(0, 3, 60),
    45 + np.linspace(0, 20, 40) + np.random.normal(0, 2, 40)
])
gc_pause = np.concatenate([
    10 + np.random.normal(0, 1, 60),
    10 + np.linspace(0, 30, 40) + np.random.normal(0, 2, 40)
])
latency = np.concatenate([
    100 + np.random.normal(0, 5, 60),
    100 + np.linspace(0, 80, 40) + np.random.normal(0, 4, 40)
])

# Thresholds cho univariate
thresholds = {'Memory (%)': 90, 'CPU (%)': 80, 'GC Pause (ms)': 50, 'Latency (ms)': 250}
metrics_data = [memory, cpu, gc_pause, latency]
metric_names = list(thresholds.keys())

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
fig.suptitle('Câu 4: Univariate MISS vs Multivariate CATCH (Memory Leak Scenario)', fontsize=13, fontweight='bold')

# === Part 1: Univariate ===
leak_zone = np.ones(n_time); leak_zone[:60] = 0
for i, (ax, metric, name) in enumerate(zip(axes.flat[:4], metrics_data, metric_names)):
    ax.plot(t_arr, metric, color='#3498DB', linewidth=1.5)
    thr = thresholds[name]
    ax.axhline(thr, color='red', linestyle='--', linewidth=2, label=f'Threshold: {thr}')
    ax.fill_between(t_arr, 0, metric, where=leak_zone==1, alpha=0.15, color='orange', label='Leak phase')
    breached = np.where(metric > thr)[0]
    if len(breached):
        ax.scatter(breached, metric[breached], color='red', s=40, zorder=5)
    alert_status = '⚠️ ALERT' if len(breached) > 0 else '✅ No Alert (MISSED!)'
    ax.set_title(f'{name} — {alert_status}', fontweight='bold')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# === Part 2: Multivariate (Isolation Forest) ===
X_multi = np.column_stack([memory, cpu, gc_pause, latency])
scaler_m = StandardScaler()
X_scaled = scaler_m.fit_transform(X_multi)
iforest_m = IsolationForest(contamination=0.15, random_state=42)
scores_m = -iforest_m.fit(X_scaled).score_samples(X_scaled)

axes[2, 0].plot(t_arr, scores_m, color='#9B59B6', linewidth=2, label='Anomaly Score')
threshold_score = np.percentile(scores_m[:60], 95)  # 95th percentile of normal period
axes[2, 0].axhline(threshold_score, color='red', linestyle='--', linewidth=2, label=f'Threshold: {threshold_score:.3f}')
axes[2, 0].axvspan(60, 100, alpha=0.2, color='orange', label='Memory Leak Phase')
detected_multi = np.where(scores_m > threshold_score)[0]
axes[2, 0].scatter(detected_multi, scores_m[detected_multi], color='red', s=40, zorder=5,
                  label=f'Detected: {len(detected_multi)} points')
axes[2, 0].set_title('Multivariate (Isolation Forest) — ⚠️ ALERT TRIGGERED!', fontweight='bold')
axes[2, 0].set_xlabel('Time Step'); axes[2, 0].legend(fontsize=8); axes[2, 0].grid(True, alpha=0.3)

# Correlation heatmap
import matplotlib.colors as mcolors
df_metrics = pd.DataFrame(X_multi[60:], columns=['Memory', 'CPU', 'GC Pause', 'Latency'])
corr = df_metrics.corr()
im = axes[2, 1].imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
axes[2, 1].set_xticks(range(4)); axes[2, 1].set_yticks(range(4))
axes[2, 1].set_xticklabels(['Memory', 'CPU', 'GC', 'Latency'], rotation=45)
axes[2, 1].set_yticklabels(['Memory', 'CPU', 'GC', 'Latency'])
plt.colorbar(im, ax=axes[2, 1])
for i in range(4):
    for j in range(4):
        axes[2, 1].text(j, i, f'{corr.values[i,j]:.2f}', ha='center', va='center', fontsize=11)
axes[2, 1].set_title('Correlation Matrix (Leak phase)\nAll metrics highly correlated!', fontweight='bold')

plt.tight_layout()
plt.savefig('images/q4_univariate_vs_multivariate.png', dpi=120, bbox_inches='tight')
plt.show()

uni_alerts = sum(1 for m, t in zip(metrics_data, thresholds.values()) if np.any(m > t))
print(f'Univariate alerts triggered: {uni_alerts}/4 metrics')
print(f'Multivariate detected {len(detected_multi)} anomalous points in leak phase')
print('✅ Câu 4 hoàn thành!')


---
## Câu 5: Precision vs Recall trong AIOps
<a id='câu-5'></a>

### Định nghĩa

$$\text{Precision} = \frac{TP}{TP + FP} \qquad \text{Recall} = \frac{TP}{TP + FN}$$

| | **Predicted Positive** | **Predicted Negative** |
|--|--|--|
| **Actual Positive** | TP (True Positive) | FN (False Negative) |
| **Actual Negative** | FP (False Positive) | TN (True Negative) |

### Tại sao AIOps ưu tiên RECALL?

**Cost asymmetry:**
- **FN (Miss anomaly):** Service down, data loss, SLA breach, revenue loss → **Chi phí cực cao**
- **FP (False alert):** Engineer mất 5-15 phút điều tra → **Annoying nhưng manageable**

**Nguyên tắc vàng:** *"Better to investigate a false alarm than miss a real incident"*

**Ví dụ thực tế:**
- DDoS attack → FN → hệ thống sập, khách hàng không truy cập được → thiệt hại hàng triệu USD
- False DDoS alert → investigate 10 phút → tốn thời gian nhưng OK

### Trade-off khi tune threshold

| Threshold | Recall | Precision | Hệ quả |
|-----------|--------|-----------|--------|
| **Thấp** | ↑ High | ↓ Low | Ít miss anomaly, nhiều false positives → alert fatigue |
| **Cao** | ↓ Low | ↑ High | Ít false alarm nhưng miss real incidents → dangerous |

**Alert Fatigue:** Khi FP quá nhiều → engineers bắt đầu **bỏ qua alert** → khi real incident xảy ra cũng bị ignore!

**Giải pháp cân bằng:**
- **F2-score:** $F_2 = \frac{5 \cdot P \cdot R}{4P + R}$ — weight Recall gấp 2 lần Precision
- **Alert deduplication:** Gộp các alert liên quan thành 1
- **Priority scoring:** Phân loại mức độ nghiêm trọng của alert
- **Multi-level thresholds:** Warning (thấp) và Critical (cao)


In [ ]:
# ── Demo: Precision-Recall Trade-off Analysis ──
from sklearn.metrics import precision_recall_curve, f1_score

np.random.seed(42)
n_samples = 1000
n_anomaly = 50

# Simulate anomaly scores
normal_scores = np.random.normal(0.3, 0.15, n_samples - n_anomaly).clip(0, 1)
anomaly_scores_sim = np.random.normal(0.7, 0.15, n_anomaly).clip(0, 1)
all_scores = np.concatenate([normal_scores, anomaly_scores_sim])
y_true = np.array([0] * (n_samples - n_anomaly) + [1] * n_anomaly)

# Precision-Recall curve
precision_vals, recall_vals, thresholds_pr = precision_recall_curve(y_true, all_scores)

# F1 and F2 scores for different thresholds
f_scores = {'F1': [], 'F2': [], 'threshold': []}
for thr in np.linspace(0.1, 0.9, 50):
    preds = (all_scores >= thr).astype(int)
    from sklearn.metrics import precision_score, recall_score
    p = precision_score(y_true, preds, zero_division=0)
    r = recall_score(y_true, preds, zero_division=0)
    f1 = 2*p*r/(p+r+1e-9)
    f2 = 5*p*r/(4*p+r+1e-9)
    f_scores['F1'].append(f1); f_scores['F2'].append(f2); f_scores['threshold'].append(thr)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Câu 5: Precision vs Recall Trade-off trong AIOps', fontsize=13, fontweight='bold')

# Plot 1: Precision-Recall Curve
axes[0].plot(recall_vals, precision_vals, color='#9B59B6', linewidth=2.5)
axes[0].fill_between(recall_vals, precision_vals, alpha=0.15, color='#9B59B6')
axes[0].axvline(0.9, color='red', linestyle='--', linewidth=2, label='Recall=0.9 target')
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve\n(AIOps muốn Recall cao)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[0].annotate('High Recall\nLow Precision\n(AIOps zone)', xy=(0.85, 0.5), fontsize=10,
                color='red', ha='center',
                arrowprops=dict(arrowstyle='->', color='red'),
                xytext=(0.65, 0.35))

# Plot 2: F1 vs F2 scores
axes[1].plot(f_scores['threshold'], f_scores['F1'], color='#3498DB', linewidth=2.5, label='F1-score (balanced)')
axes[1].plot(f_scores['threshold'], f_scores['F2'], color='#E74C3C', linewidth=2.5, label='F2-score (recall weighted)')
best_f1 = f_scores['threshold'][np.argmax(f_scores['F1'])]
best_f2 = f_scores['threshold'][np.argmax(f_scores['F2'])]
axes[1].axvline(best_f1, color='#3498DB', linestyle=':', label=f'Best F1 threshold: {best_f1:.2f}')
axes[1].axvline(best_f2, color='#E74C3C', linestyle=':', label=f'Best F2 threshold: {best_f2:.2f}')
axes[1].set_xlabel('Threshold'); axes[1].set_ylabel('Score')
axes[1].set_title('F1 vs F2 Score\n(F2 optimal threshold thấp hơn = Recall ưu tiên)')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

# Plot 3: Alert fatigue vs Miss incident cost
thresholds_cost = np.linspace(0.1, 0.9, 100)
recall_cost = []
fp_cost = []
for thr in thresholds_cost:
    preds = (all_scores >= thr).astype(int)
    r = np.sum((preds==1) & (y_true==1)) / (np.sum(y_true==1) + 1e-9)
    fp = np.sum((preds==1) & (y_true==0))
    recall_cost.append(r)
    fp_cost.append(fp)

ax3b = axes[2].twinx()
l1, = axes[2].plot(thresholds_cost, [1-r for r in recall_cost], color='#E74C3C', linewidth=2.5, label='Miss rate (1-Recall)')
l2, = ax3b.plot(thresholds_cost, fp_cost, color='#3498DB', linewidth=2.5, linestyle='--', label='False Positives count')
axes[2].axvspan(0.2, 0.45, alpha=0.1, color='green', label='Recommended zone')
axes[2].set_xlabel('Threshold'); axes[2].set_ylabel('Miss Rate', color='#E74C3C')
ax3b.set_ylabel('False Positives', color='#3498DB')
axes[2].set_title('Trade-off: Miss Rate vs False Positives\n(Green = recommended AIOps zone)')
axes[2].legend(handles=[l1, l2, mpatches.Patch(color='green', alpha=0.3, label='Recommended')], fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('images/q5_precision_recall.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Best F1 threshold: {best_f1:.2f} | Best F2 threshold: {best_f2:.2f}')
print('✅ Câu 5 hoàn thành!')


---
## Ảnh Knowledge Check Viết Tay
<a id='ảnh-viết-tay'></a>

Dưới đây là ảnh bài làm viết tay cho từng câu hỏi:


In [ ]:
# ── Hiển thị ảnh knowledge check viết tay ──
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

handwritten_images = [
    ('images/knowledge_check_page1.png', 'Câu 1: Skewness & 3σ'),
    ('images/knowledge_check_page2.png', 'Câu 2: 3σ vs EWMA vs STL'),
    ('images/knowledge_check_page3.png', 'Câu 3: Isolation Forest'),
    ('images/knowledge_check_page4.png', 'Câu 4: Univariate vs Multivariate'),
    ('images/knowledge_check_page5.png', 'Câu 5: Precision vs Recall'),
]

fig, axes = plt.subplots(1, 5, figsize=(25, 8))
fig.suptitle('Knowledge Check — Ảnh Viết Tay', fontsize=16, fontweight='bold')

for ax, (img_path, title) in zip(axes, handwritten_images):
    if os.path.exists(img_path):
        img = mpimg.imread(img_path)
        ax.imshow(img)
        ax.set_title(title, fontsize=9, fontweight='bold')
    else:
        ax.text(0.5, 0.5, 'Image\nnot found', ha='center', va='center')
        ax.set_title(title, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('images/knowledge_check_all.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Tất cả ảnh knowledge check đã được hiển thị!')


---
## Tổng kết

| Câu | Nội dung | Status |
|-----|----------|--------|
| Câu 1 | Skewness, 3σ fail với skewed data, Log transform + Robust IQR | ✅ |
| Câu 2 | 3σ vs EWMA vs STL: detect type, failure mode, use case | ✅ |
| Câu 3 | Isolation Forest path length, feature engineering importance | ✅ |
| Câu 4 | Memory leak: univariate miss, multivariate catch | ✅ |
| Câu 5 | Recall priority in AIOps, threshold trade-off, F2-score | ✅ |

**Repository:** https://github.com/KimDung1/aiops_lekimdung  
**Submitted by:** Lê Kim Dung — 01/06/2026
